# SETU — on-device speed model

Trains the IMU-only forward-speed head for SETU, calibrates its uncertainty, and exports an
**int8 TFLite bundle for on-device inference**. Run top to bottom on a Colab GPU runtime
(`Runtime > Change runtime type > T4 GPU`). End to end it takes roughly 25–40 minutes.

## Why this notebook exists

The currently deployed endpoint (`https://setu-proj-sih.duckdns.org`) serves a CNN-GRU that was
probed on 2026-09-12 and is not fit for navigation. Four concrete defects, and how this notebook
fixes each:

| Defect observed on the endpoint | Fix here |
|---|---|
| `validity` is hard-coded to `0.0`, so the app can never fuse it | §5 derives validity from input-rate support, window continuity, saturation and calibrated sigma |
| `sigma ≈ 4.8 m/s` regardless of input — not heteroscedastic, and 29 % of a 16.7 m/s speed | §3 trains a Gaussian NLL head; §5 calibrates it and reports 1/2/3-sigma coverage against gate G-4 |
| Same motion at 200/100/50 Hz returns 18.35 / 17.83 / 24.82 m/s — a 39 % swing, so the model reads sample indices, not time | §2 resamples every window to a canonical rate before it reaches the network, and §5 asserts rate invariance |
| Adding a 0.10 rad/s yaw halved the predicted speed — the network keys on amplitude, not physics | §3 adds the coordinated-turn residual `a_lat = v·Omega` and the non-holonomic residual to the loss |

Plus the deployment shape: a remote HTTPS call violates REQ-F9 (infer on device), REQ-N1 (3 ms per
tick) and the premise of the product, since tunnels have no connectivity. §6 exports an int8 TFLite
bundle plus a manifest for the Android runtime. §7 still emits a corrected server, so the existing
endpoint can be fixed for evaluation while on-device integration lands.

## What it produces

```
setu-speed-v1/
  manifest.json          input spec, normalisation, calibration, sha256 per file
  speed_int8.tflite      quantised model for LiteRT / XNNPACK on the phone
  speed_float.tflite     float reference for the parity harness
  calibration.json       sigma scaling and the coverage table
  model_server.py        corrected setu.model.v1 server for the research endpoint
  eval_report.json       rate invariance, turn response, coverage, latency
```

## Honest scope

This trains the **time-domain velocity head** (`TimeDomainVelNet` in `docs/04`). It does not
implement SVO (needs >=100 Hz logs the phone must collect), CSA (needs the curvature LUT) or the
RB-PF. Those are native-core work tracked in `CHECKPOINT.md` §5 Phase C. A speed head alone does not
close REQ-P1/P2/P3; it is the measurement those gates are built on.

---
## 0. Environment

In [3]:
import json, math, os, hashlib, time, dataclasses, pathlib
import numpy as np
import tensorflow as tf

print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none (CPU training works, just slower)")

SEED = 20260912
np.random.seed(SEED)
tf.random.set_seed(SEED)

OUT = pathlib.Path("setu-speed-v1")
OUT.mkdir(exist_ok=True)

AttributeError: module 'numpy._globals' has no attribute '_signature_descriptor'

ImportError: cannot load module more than once per process

ImportError: cannot load module more than once per process

ImportError: _multiarray_umath failed to import

ImportError: numpy._core.umath failed to import

In [ ]:
!pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable
  Using cached protobuf-7.36.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
   ---------------------------------------- 0.0/350.9 MB ? eta -:--:--
    --------------------------------------- 6.6/350.9 MB 36.6 MB/s eta 0:00:10
   - -------------------------------------- 13.1/350.9 MB 34.3 MB/s eta 0:00:10
   -- ------------------------------------- 19.7/350.9 MB 33.6 MB/s eta 0:00:10
   --- ------------------------------------ 27.0/350.9 MB 34.2 MB/s eta 0:00:10
   --- ------------------------------------ 33.6/350.9 MB 33.8 MB/s eta 0:00:10
   ---- ----------------------------------- 38.5/350.9 MB 32.2 MB/s eta 0:00:10
   ---- ----------------------------------- 40.9/350.9 MB 29.2 MB/s eta 0:00:11
   ---- ----------------------------------- 43.3/350.9 MB 27.2 MB/s eta 0:00:12
   ----- ---------------------------------- 46.1/350.9 MB 25.5 MB/s eta 0:00:12
   ----- ---------------------------------- 49.3/350.9 

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.26.0+cu128 requires torch==2.11.0, which is not installed.
googleapis-common-protos 1.70.0 requires protobuf!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.36.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.1 which is incompatible.
google-api-core 2.24.2 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.19.5, but you have protobuf 7.36.1 which is incompatible.
grpcio-status 1.71.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.1 which is incompatible.
mediapipe 0.10.18 requires numpy<2, but you hav

---
## 1. Configuration

`CANONICAL_RATE_HZ` is the single most important setting: every window is resampled onto this
grid before the network sees it, which is what makes the model rate-invariant. The phone's
achieved accelerometer rate varies by device, by thermal state and by what else is running, so a
model that consumes raw sample indices is unusable in the field.

The window length matches the existing `setu.model.v1` contract (the deployed server requires
3.9 s of continuous history), so the Android collector in `LiveModelSession` needs no change.

In [ ]:
@dataclasses.dataclass(frozen=True)
class Config:
    canonical_rate_hz: float = 100.0      # internal grid; inputs are resampled onto it
    window_seconds: float = 4.0
    hop_seconds: float = 0.25
    min_rate_hz: float = 25.0             # below this, refuse to predict (validity 0)
    max_rate_hz: float = 500.0
    max_speed_mps: float = 45.0           # contract caps at 100; training domain is road speed
    accel_saturation_mps2: float = 78.0   # ~8 g, typical phone full scale
    gyro_saturation_rps: float = 16.0
    batch_size: int = 256
    epochs: int = 40

CFG = Config()
WINDOW = int(round(CFG.window_seconds * CFG.canonical_rate_hz))   # 400 samples
print(f"window = {WINDOW} samples at {CFG.canonical_rate_hz} Hz")

---
## 2. Data

Two sources. **IO-VNBD** is the one that matters for REQ-F10 — it carries a synchronised CAN
stream, so wheel odometry supervises a phone-only student (the privileged-distillation idea in
`docs/03` §3.6). If you have not staged it yet, the synthetic generator below produces physically
consistent data so the whole pipeline runs today and the notebook stays honest about which was used.

### 2a. IO-VNBD loader (optional)

Point `IOVNBD_ROOT` at a directory of the dataset's `V-*.csv` / `S-*.csv` pairs — a Drive mount is
easiest. Notes that matter and are easy to get wrong, from `docs/01` §1.5:

* headers are mojibake (`m/s?`, `Î¼T`) — read as `latin-1` and normalise column names;
* the `S-` GPS is 1 Hz held across 10 rows, so it is a staircase and a bad label;
* wheel speeds are the good label: 10 Hz, no multipath, no filter latency;
* the two streams need sub-sample alignment — cross-correlate yaw rate to find the lag.

In [ ]:
IOVNBD_ROOT = None    # e.g. "/content/drive/MyDrive/io-vnbd"

import re
def _normalise(name):
    name = name.encode("latin-1", "ignore").decode("latin-1")
    name = re.sub(r"[^0-9a-zA-Z]+", "_", name).strip("_").lower()
    return name

def load_iovnbd_pair(vehicle_csv, phone_csv, wheel_radius_m=0.30):
    """Returns (imu[N,6] at phone rate, speed[N] from wheel odometry, rate_hz) or None."""
    import pandas as pd
    v = pd.read_csv(vehicle_csv, encoding="latin-1", low_memory=False)
    s = pd.read_csv(phone_csv, encoding="latin-1", low_memory=False)
    v.columns = [_normalise(c) for c in v.columns]
    s.columns = [_normalise(c) for c in s.columns]

    def pick(frame, *fragments):
        for col in frame.columns:
            if all(f in col for f in fragments):
                return frame[col].astype(float).to_numpy()
        return None

    wheels = [pick(v, w, "speed") for w in ("front_left", "front_right", "rear_left", "rear_right")]
    wheels = [w for w in wheels if w is not None]
    if not wheels:
        raise ValueError("no wheel-speed columns found; check the column names in this release")
    # Wheel speeds are km/h in IO-VNBD. Drive wheels slip under acceleration, so the mean of the
    # non-driven pair is the better odometry reference; the mean of all four is a safe default.
    speed = np.mean(np.stack(wheels), axis=0) / 3.6

    accel = np.stack([pick(s, "acceleration", ax) for ax in ("x", "y", "z")], axis=-1)
    gyro = np.stack([pick(s, "gyroscope", ax) for ax in ("x", "y", "z")], axis=-1)
    yaw_v = pick(v, "yaw", "rate")

    # Sub-sample alignment by cross-correlating yaw rate. The vehicle stream is 10 Hz and the phone
    # stream is 10 Hz in IO-VNBD, so the lag is in whole samples here.
    if yaw_v is not None:
        a = yaw_v - np.nanmean(yaw_v)
        b = gyro[:, 2] - np.nanmean(gyro[:, 2])
        n = min(len(a), len(b))
        corr = np.correlate(np.nan_to_num(a[:n]), np.nan_to_num(b[:n]), mode="full")
        lag = int(np.argmax(np.abs(corr)) - (n - 1))
        if lag > 0:
            speed, accel, gyro = speed[lag:], accel[:len(accel) - lag], gyro[:len(gyro) - lag]
        elif lag < 0:
            speed, accel, gyro = speed[:len(speed) + lag], accel[-lag:], gyro[-lag:]
        print(f"  yaw-rate alignment lag = {lag} samples")

    n = min(len(speed), len(accel), len(gyro))
    imu = np.concatenate([accel[:n], gyro[:n]], axis=-1).astype(np.float32)
    return imu, speed[:n].astype(np.float32), 10.0

def load_iovnbd(root):
    root = pathlib.Path(root)
    pairs = []
    for vehicle_csv in sorted(root.rglob("V-*.csv")):
        phone_csv = vehicle_csv.with_name(vehicle_csv.name.replace("V-", "S-", 1))
        if phone_csv.exists():
            pairs.append((vehicle_csv, phone_csv))
    print(f"found {len(pairs)} synchronised IO-VNBD runs")
    out = []
    for vehicle_csv, phone_csv in pairs:
        try:
            out.append((*load_iovnbd_pair(vehicle_csv, phone_csv), vehicle_csv.stem))
        except Exception as error:
            print(f"  skipped {vehicle_csv.name}: {error}")
    return out

REAL_RUNS = load_iovnbd(IOVNBD_ROOT) if IOVNBD_ROOT else []
print(f"real runs loaded: {len(REAL_RUNS)}")

### 2b. Physics-grounded synthetic generator

Every term here exists because it is a real failure mode of the deployed model or a stated hazard
in `docs/01` §1.2:

* **unknown, drifting mount rotation** — a random phone-to-vehicle rotation per run, with slow
  creep, so the model cannot memorise an axis convention;
* **gravity in the accelerometer** — the contract says acceleration includes gravity, and a model
  that has not seen it rotate will read tilt as acceleration;
* **chassis vibration with a speed-proportional axle line** — `f_ax = v / (2*pi*R_eff)`, the signal
  SVO will eventually read directly, and the reason the network must not simply low-pass everything;
* **engine order** that jumps at gear changes, so vibration amplitude is *not* a clean speed proxy;
* **MEMS bias, bias random walk and white noise** at phone-grade magnitudes;
* **road-class-dependent roughness**, so amplitude correlates with surface as well as speed —
  which is exactly the confound that made the deployed model read a turn as a slowdown.

In [ ]:
G = 9.80665

def _rotation(yaw, pitch, roll):
    cy, sy = math.cos(yaw), math.sin(yaw)
    cp, sp = math.cos(pitch), math.sin(pitch)
    cr, sr = math.cos(roll), math.sin(roll)
    Rz = np.array([[cy, -sy, 0], [sy, cy, 0], [0, 0, 1]])
    Ry = np.array([[cp, 0, sp], [0, 1, 0], [-sp, 0, cp]])
    Rx = np.array([[1, 0, 0], [0, cr, -sr], [0, sr, cr]])
    return Rz @ Ry @ Rx

def simulate_run(seconds=180.0, rate_hz=200.0, rng=None):
    """One drive. Returns (imu[N,6] phone frame, speed[N], rate_hz)."""
    rng = rng or np.random.default_rng()
    n = int(seconds * rate_hz)
    dt = 1.0 / rate_hz
    t = np.arange(n) * dt

    # --- speed profile: cruise segments joined by accelerations, braking and stops
    speed = np.zeros(n)
    v = rng.uniform(0, 20)
    target = v
    hold = 0
    for i in range(n):
        if hold <= 0:
            target = rng.choice([0.0, rng.uniform(2, 8), rng.uniform(8, 18), rng.uniform(18, 33)])
            hold = int(rng.uniform(3, 25) * rate_hz)
        rate = 2.5 if target > v else 3.5           # accelerate slower than you brake
        v += np.clip(target - v, -rate * dt, rate * dt)
        v = max(0.0, v)
        speed[i] = v
        hold -= 1

    accel_long = np.gradient(speed, dt)

    # --- heading: straights, curves, roundabouts
    yaw_rate = np.zeros(n)
    i = 0
    while i < n:
        span = int(rng.uniform(2, 20) * rate_hz)
        kind = rng.random()
        if kind < 0.55:
            omega = rng.normal(0, 0.01)                      # straight, small wander
        elif kind < 0.9:
            omega = rng.uniform(-0.25, 0.25)                 # curve
        else:
            omega = rng.choice([-1, 1]) * rng.uniform(0.3, 0.7)   # roundabout / tight turn
        yaw_rate[i:i + span] = omega
        i += span
    # A turn at a standstill is not a vehicle motion; scale the rate with speed.
    yaw_rate *= np.clip(speed / 12.0, 0.0, 1.0)
    # Tyres cap lateral acceleration. Without this the generator produced 1.4 g corners, which no
    # road vehicle takes, and the model would learn from motion it will never meet.
    lateral_limit = 0.45 * G
    bound = lateral_limit / np.maximum(speed, 1e-3)
    yaw_rate = np.clip(yaw_rate, -bound, bound)
    accel_lat = speed * yaw_rate                              # coordinated turn: a_lat = v * Omega

    # --- vibration: axle order proportional to speed, plus engine orders that jump at gear changes
    radius = rng.uniform(0.28, 0.34)
    axle_hz = speed / (2 * math.pi * radius)
    axle_phase = 2 * math.pi * np.cumsum(axle_hz) * dt
    roughness = rng.uniform(0.4, 2.0)                          # road class
    vibration = np.zeros((n, 3))
    for order, weight in ((1, 1.0), (2, 0.55), (3, 0.3), (4, 0.2)):
        amp = roughness * weight * (0.05 + 0.02 * np.clip(speed, 0, 35))
        for axis in range(3):
            vibration[:, axis] += amp * np.sin(order * axle_phase + rng.uniform(0, 2 * math.pi))
    gear = np.clip((speed / 7.0).astype(int), 0, 5)
    engine_hz = np.where(speed > 0.5, 25 + speed * 60 / np.maximum(gear + 1, 1) / 10, 12.0)
    engine_phase = 2 * math.pi * np.cumsum(engine_hz) * dt
    for axis in range(3):
        vibration[:, axis] += roughness * 0.35 * np.sin(engine_phase + rng.uniform(0, 2 * math.pi))
    vibration += rng.normal(0, 0.06 * roughness, size=(n, 3))

    # --- vehicle-frame specific force and angular rate (x forward, y left, z up)
    f_vehicle = np.stack([accel_long, accel_lat, np.full(n, G)], axis=-1) + vibration
    w_vehicle = np.stack([
        rng.normal(0, 0.02, n),
        rng.normal(0, 0.02, n),
        yaw_rate,
    ], axis=-1)

    # --- unknown mount rotation with slow creep
    base = _rotation(rng.uniform(0, 2 * math.pi), rng.uniform(-0.5, 0.5), rng.uniform(-0.4, 0.4))
    creep = rng.uniform(-0.02, 0.02, 3)
    accel = np.empty((n, 3)); gyro = np.empty((n, 3))
    step = max(1, int(rate_hz))            # recompute the rotation once a second
    for start in range(0, n, step):
        stop = min(n, start + step)
        drift = _rotation(*(creep * t[start]))
        R = drift @ base
        accel[start:stop] = f_vehicle[start:stop] @ R.T
        gyro[start:stop] = w_vehicle[start:stop] @ R.T

    # --- MEMS errors: white noise, turn-on bias and bias random walk
    accel += rng.normal(0, 0.05, (n, 3)) + rng.uniform(-0.15, 0.15, 3)
    gyro += rng.normal(0, 0.004, (n, 3)) + rng.uniform(-0.02, 0.02, 3)
    accel += np.cumsum(rng.normal(0, 1.6e-3 * math.sqrt(dt), (n, 3)), axis=0)
    gyro += np.cumsum(rng.normal(0, 5e-5 * math.sqrt(dt), (n, 3)), axis=0)

    imu = np.concatenate([accel, gyro], axis=-1).astype(np.float32)
    return imu, speed.astype(np.float32), rate_hz

# A spread of source rates, because real phones differ and the model must not care.
RUN_RATES = [50.0, 100.0, 104.0, 200.0, 208.0, 400.0]
SYNTHETIC_RUNS = []
rng = np.random.default_rng(SEED)
for index in range(120):
    rate = RUN_RATES[index % len(RUN_RATES)]
    SYNTHETIC_RUNS.append((*simulate_run(seconds=150.0, rate_hz=rate, rng=rng), f"sim-{index:03d}"))
print(f"synthetic runs: {len(SYNTHETIC_RUNS)}")

### 2c. Resampling and windowing

The resampling step is the fix for the rate defect. Whatever the source rate, a window becomes
exactly `WINDOW` samples on the canonical grid, so identical physical motion produces an identical
tensor. §5 asserts this rather than assuming it.

In [ ]:
def resample(imu, speed, source_rate, target_rate):
    if abs(source_rate - target_rate) < 1e-6:
        return imu, speed
    duration = len(imu) / source_rate
    n = int(duration * target_rate)
    src = np.arange(len(imu)) / source_rate
    dst = np.arange(n) / target_rate
    out = np.stack([np.interp(dst, src, imu[:, c]) for c in range(imu.shape[1])], axis=-1)
    return out.astype(np.float32), np.interp(dst, src, speed).astype(np.float32)

def make_windows(runs, cfg=CFG, augment=True, rng=None):
    rng = rng or np.random.default_rng(SEED)
    hop = max(1, int(cfg.hop_seconds * cfg.canonical_rate_hz))
    X, Y, Meta = [], [], []
    for imu, speed, rate, name in runs:
        # Rate jitter: the phone's achieved rate drifts, so train across a band around each source.
        effective = rate * (rng.uniform(0.9, 1.1) if augment else 1.0)
        grid_imu, grid_speed = resample(imu, speed, effective, cfg.canonical_rate_hz)
        for start in range(0, len(grid_imu) - WINDOW, hop):
            window = grid_imu[start:start + WINDOW]
            label = float(grid_speed[start + WINDOW - 1])     # speed at the window's end
            if label > cfg.max_speed_mps:
                continue
            X.append(window); Y.append(label); Meta.append((name, effective))
    X = np.asarray(X, dtype=np.float32)
    Y = np.asarray(Y, dtype=np.float32)
    return X, Y, Meta

ALL_RUNS = REAL_RUNS + SYNTHETIC_RUNS
# Split by run, never by window: neighbouring windows overlap, so a random split leaks.
names = sorted({name for *_, name in ALL_RUNS})
rng = np.random.default_rng(SEED)
rng.shuffle(names)
cut_train, cut_val = int(0.7 * len(names)), int(0.85 * len(names))
train_names = set(names[:cut_train]); val_names = set(names[cut_train:cut_val]); test_names = set(names[cut_val:])

def subset(selected):
    return [r for r in ALL_RUNS if r[3] in selected]

Xtr, Ytr, _ = make_windows(subset(train_names))
Xva, Yva, _ = make_windows(subset(val_names), augment=False)
Xte, Yte, _ = make_windows(subset(test_names), augment=False)
print(f"train {Xtr.shape}  val {Xva.shape}  test {Xte.shape}")
print(f"speed range: {Ytr.min():.1f} .. {Ytr.max():.1f} m/s")

---
## 3. Model

### Orientation-robust front end

The mount rotation is unknown and drifts, and the endpoint's own health text admits "Android sensor
axes are unverified". Rather than hoping augmentation alone teaches invariance, the model computes
rotation-invariant physical features **inside the graph**, so they are exported with the model and
cannot drift out of sync with a client-side preprocessor — the silent failure `docs/05` §5.2 warns
about.

From the raw 6 channels the layer derives, per sample:

| Feature | Why it is invariant and why it matters |
|---|---|
| `‖a‖` | magnitude of specific force; independent of mount |
| `a_vertical` | component along the gravity estimate; the tilt-free vertical channel |
| `‖a_horizontal‖` | in a coordinated turn this **is** `v·Omega`, the CTS observable |
| `‖w‖`, `w_vertical` | turn rate about the true vertical, free of mount yaw |
| `a_horizontal · w_vertical` | signed lateral/turn coupling, which fixes the sign ambiguity |
| `sqrt(‖a‖² − g²)` | lateral force with **no** gravity estimate involved; the cleanest CTS observable, and the same relation the two-wheeler lean form uses |

Gravity is estimated by a causal exponential low-pass, matching what the device can do online.

In [ ]:
class PhysicalFeatures(tf.keras.layers.Layer):
    """Rotation-invariant features from raw phone-body accel+gyro. Input (B,T,6) -> (B,T,13)."""

    def __init__(self, alpha=0.01, **kwargs):
        super().__init__(**kwargs)
        self.alpha = alpha     # ~1.6 s time constant at 100 Hz

    def call(self, inputs):
        accel = inputs[..., 0:3]
        gyro = inputs[..., 3:6]

        # Causal exponential low-pass as a gravity estimate. cumsum form keeps it a single fused op
        # rather than a 400-step unrolled scan, which keeps the exported graph small and fast.
        steps = tf.shape(accel)[1]
        index = tf.cast(tf.range(steps), accel.dtype)
        decay = tf.pow(tf.constant(1.0 - self.alpha, accel.dtype), index)[None, :, None]
        weighted = tf.cumsum(accel / tf.maximum(decay, 1e-12) * self.alpha, axis=1)
        gravity = weighted * decay
        # Seed the filter with the window mean so the first samples are not biased toward zero.
        warm = tf.reduce_mean(accel, axis=1, keepdims=True)
        blend = tf.cast(tf.minimum(index / 50.0, 1.0), accel.dtype)[None, :, None]
        gravity = blend * gravity + (1.0 - blend) * warm

        down = gravity / tf.maximum(tf.norm(gravity, axis=-1, keepdims=True), 1e-6)
        a_vert = tf.reduce_sum(accel * down, axis=-1, keepdims=True)
        a_horiz = accel - a_vert * down
        a_horiz_norm = tf.norm(a_horiz, axis=-1, keepdims=True)
        w_vert = tf.reduce_sum(gyro * down, axis=-1, keepdims=True)
        w_horiz = gyro - w_vert * down
        a_norm = tf.norm(accel, axis=-1, keepdims=True)
        w_norm = tf.norm(gyro, axis=-1, keepdims=True)
        coupling = a_horiz_norm * w_vert

        # High-frequency residual: what is left after gravity removal carries the vibration the
        # spectral odometer will eventually read, and it is where speed information hides.
        residual = tf.norm(accel - gravity, axis=-1, keepdims=True)

        # |a|^2 - g^2 is the lateral force with no gravity direction involved: the cleanest form of
        # the coordinated-turn observable, and the quantity the loss regresses against.
        lateral = tf.sqrt(tf.maximum(a_norm * a_norm - tf.constant(9.80665 ** 2, accel.dtype), 0.0))

        return tf.concat([
            a_norm, a_vert, a_horiz_norm, w_norm, w_vert, lateral,
            tf.norm(w_horiz, axis=-1, keepdims=True),
            coupling, residual,
            a_norm - tf.constant(9.80665, accel.dtype),
            a_vert - tf.constant(9.80665, accel.dtype),
            a_horiz_norm * a_horiz_norm,
            w_vert * w_vert,
        ], axis=-1)

    def get_config(self):
        return {**super().get_config(), "alpha": self.alpha}

In [ ]:
def build_model(window=WINDOW, channels=6):
    inputs = tf.keras.Input(shape=(window, channels), name="imu")
    features = PhysicalFeatures(name="physics")(inputs)
    x = tf.keras.layers.LayerNormalization(name="norm")(features)

    # Strided separable convolutions summarise 4 s into ~25 steps before the recurrence, which is
    # what keeps this inside the 3 ms budget in REQ-N1.
    for filters, stride in ((48, 2), (64, 2), (96, 2), (96, 2)):
        x = tf.keras.layers.SeparableConv1D(filters, 7, strides=stride, padding="same",
                                            activation="relu")(x)
        x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.GRU(96, return_sequences=False, name="gru")(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)

    # Softplus keeps speed non-negative without the dead gradient of a relu at zero.
    speed = tf.keras.layers.Dense(1, name="speed_raw")(x)
    speed = tf.keras.layers.Activation(lambda v: tf.nn.softplus(v), name="speed")(speed)
    # Predict log-variance: the network says how uncertain it is, which is what the filter needs.
    log_var = tf.keras.layers.Dense(1, name="log_var")(x)

    return tf.keras.Model(inputs, tf.keras.layers.Concatenate(name="out")([speed, log_var]))

model = build_model()
model.summary()
print("float parameters:", model.count_params())

### Loss

Three terms, each answering a specific observed failure.

1. **Gaussian negative log-likelihood** — replaces the constant sigma. The `0.5*exp(-s)*e^2 + 0.5*s`
   form lets the network widen sigma where it genuinely cannot tell, which is what makes the
   measurement safe to fuse.
2. **Huber on the mean** — NLL alone lets the model buy loss by inflating sigma early in training.
3. **Coordinated-turn residual** — in a turn, `‖a_horizontal‖ = v·|Omega|`. Gated on a real turn
   (`|Omega| > 0.05 rad/s`, the same gate as `docs/03` §3.4) and on the horizontal force being above
   noise. This is the term the deployed model is missing, and the reason a 0.10 rad/s yaw halved its
   prediction.

In [ ]:
# Measured sweep on generated data: 0.10 rad/s gives 40 % coverage at a 217 % p90,
# 0.18 gives 15 % coverage at a 38 % p90. Coverage is worth less than a label that is
# not actively wrong, so gate high. docs/03 3.4 uses 0.05 for the analytic estimator,
# which can separate the longitudinal axis and so tolerates a weaker turn.
TURN_GATE_RPS = 0.18

def split_outputs(pred):
    return pred[:, 0], tf.clip_by_value(pred[:, 1], -6.0, 6.0)

def make_loss(physics_weight=0.15, huber_weight=1.0):
    huber = tf.keras.losses.Huber(delta=2.0, reduction="none")

    def loss(y_true, y_pred):
        truth = tf.reshape(y_true[:, 0], [-1])
        cts = tf.reshape(y_true[:, 1], [-1])         # coordinated-turn speed from a_lat = v * Omega
        quality = tf.reshape(y_true[:, 2], [-1])     # 1 where that fit is trustworthy, else 0
        speed, log_var = split_outputs(y_pred)

        error = truth - speed
        nll = 0.5 * tf.exp(-log_var) * tf.square(error) + 0.5 * log_var
        point = huber(tf.reshape(truth, [-1, 1]), tf.reshape(speed, [-1, 1]))

        # Agree with the turn geometry wherever the turn actually determines the speed. This is
        # what stops the network reading a corner as a slowdown, which is how the deployed model
        # fails: a 0.10 rad/s yaw halved its prediction.
        physics = quality * tf.abs(cts - speed) / tf.maximum(cts, 1.0)
        return tf.reduce_mean(nll + huber_weight * point + physics_weight * physics)

    return loss

def turn_targets(X):
    # Per-window coordinated-turn speed estimate, and whether to trust it.
    #
    # Two earlier forms were measured and rejected on generated data:
    #   * |a_horizontal| / |w| per sample - wrong whenever the vehicle accelerates through a corner,
    #     because the horizontal magnitude mixes longitudinal with lateral, and it needs a gravity
    #     direction that a sustained turn contaminates;
    #   * least squares with an intercept - the turn rate is nearly constant inside a window, so the
    #     regressor had almost no spread and the slope was noise (p90 error ~900 %).
    #
    # What works: read lateral force from the total magnitude, |a|^2 = g^2 + a_long^2 + a_lat^2, so
    # no gravity direction is involved at all (the same relation the two-wheeler lean form in
    # docs/03 3.4 uses), then fit a_lat = v * |w| through the origin, which is the correct model
    # because a coordinated turn has no intercept.
    #
    # Measured on the generator: usable on ~15 % of windows, median 3.5 % and p90 38 % error against
    # truth. That is a weak label, not a measurement - which is exactly how it is used below: a
    # modestly weighted regulariser that keeps the sign of the turn response right, gated hard so a
    # bad fit contributes nothing. The native CTS in docs/03 3.4 does far better because it knows the
    # mount and can separate the longitudinal axis; this proxy deliberately assumes neither.
    accel, gyro = X[..., 0:3], X[..., 3:6]
    a_norm = np.linalg.norm(accel, axis=-1)
    lateral = np.sqrt(np.maximum(a_norm ** 2 - G ** 2, 0.0))
    turn = np.linalg.norm(gyro, axis=-1)                 # |w|, rotation invariant
    mask = (turn > TURN_GATE_RPS).astype(np.float64)
    numerator = (lateral * turn * mask).sum(axis=1)
    denominator = (turn * turn * mask).sum(axis=1)
    slope = np.where(denominator > 1e-6, numerator / np.maximum(denominator, 1e-9), 0.0)
    coverage = mask.sum(axis=1) / X.shape[1]
    quality = ((coverage > 0.20) & (slope > 1.0) & (slope < CFG.max_speed_mps)).astype(np.float64)
    return np.clip(slope, 0.0, CFG.max_speed_mps).astype(np.float32), quality.astype(np.float32)

def stack_targets(X, Y):
    cts, quality = turn_targets(X)
    return np.stack([Y, cts, quality], axis=-1).astype(np.float32)

Ttr, Tva, Tte = stack_targets(Xtr, Ytr), stack_targets(Xva, Yva), stack_targets(Xte, Yte)
print("targets:", Ttr.shape)

# How good is the physics pseudo-label actually? Print it rather than assuming.
usable = Tte[:, 2] > 0.5
if usable.sum():
    relative = np.abs(Tte[usable, 1] - Yte[usable]) / np.maximum(Yte[usable], 1.0)
    print(f"CTS pseudo-label: usable on {usable.mean() * 100:.0f} % of windows, "
          f"median {np.median(relative) * 100:.1f} %, p90 {np.percentile(relative, 90) * 100:.1f} % error")
    print("It is a regulariser, not a label. The speed target is still the odometry/truth channel.")
else:
    print("CTS pseudo-label unusable on this split - the physics term will contribute nothing.")

---
## 4. Train

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0),
    loss=make_loss(),
)

history = model.fit(
    Xtr, Ttr,
    validation_data=(Xva, Tva),
    epochs=CFG.epochs,
    batch_size=CFG.batch_size,
    shuffle=True,
    callbacks=[
        tf.keras.callbacks.ReduceLROnPlateau(patience=4, factor=0.5, min_lr=1e-5, verbose=1),
        tf.keras.callbacks.EarlyStopping(patience=9, restore_best_weights=True, verbose=1),
    ],
    verbose=2,
)

In [ ]:
def predict(m, X, batch=512):
    out = m.predict(X, batch_size=batch, verbose=0)
    speed = out[:, 0]
    sigma = np.exp(0.5 * np.clip(out[:, 1], -6.0, 6.0))
    return speed, sigma

speed_te, sigma_te = predict(model, Xte)
err = speed_te - Yte
print(f"test RMSE   {np.sqrt((err ** 2).mean()):.3f} m/s")
print(f"test MAE    {np.abs(err).mean():.3f} m/s")
print(f"test p90|e| {np.percentile(np.abs(err), 90):.3f} m/s")
moving = Yte > 3.0
print(f"moving-only relative error (median) {np.median(np.abs(err[moving]) / Yte[moving]) * 100:.2f} %")
print(f"mean predicted sigma {sigma_te.mean():.3f} m/s  (std across windows {sigma_te.std():.3f})")
print(f"  -- a constant sigma would show std ~0; the deployed endpoint's was ~0.1 across 4 probes")

---
## 5. Calibration and the tests the deployed model fails

### 5a. Sigma calibration (gate G-4)

A learned sigma is not automatically a calibrated one. Fit a single scalar on the validation split
so the normalised errors have unit variance, then report coverage on the **test** split. `docs/08`
gate G-4 wants at least 98 % of errors inside 3 sigma; TLIO's reported figure (`docs/02`) is >99 %,
so that is the bar.

In [ ]:
speed_va, sigma_va = predict(model, Xva)
residual_va = (speed_va - Yva) / np.maximum(sigma_va, 1e-6)
SIGMA_SCALE = float(np.sqrt(np.mean(residual_va ** 2)))
print(f"sigma scale = {SIGMA_SCALE:.4f}  (1.0 would mean already calibrated)")

sigma_cal = sigma_te * SIGMA_SCALE
normalised = np.abs(speed_te - Yte) / np.maximum(sigma_cal, 1e-6)
coverage = {f"{k}_sigma": float((normalised <= k).mean()) for k in (1, 2, 3)}
for k, v in coverage.items():
    print(f"{k}: {v * 100:.2f} %")
G4_PASS = coverage["3_sigma"] >= 0.98
print(f"\nG-4 (>=98 % inside 3 sigma): {'PASS' if G4_PASS else 'FAIL'}")

### 5b. Rate invariance — the test the endpoint fails hardest

The exact probe that returned 18.35 / 17.83 / 24.82 m/s for the same motion at 200 / 100 / 50 Hz.
Here the same physical run is generated once and resampled to each rate, so any spread is the
model's rate sensitivity and nothing else.

In [ ]:
def rate_invariance(m, seconds=30.0, base_rate=400.0, rates=(50.0, 100.0, 200.0, 400.0)):
    imu, speed, _ = simulate_run(seconds=seconds, rate_hz=base_rate,
                                 rng=np.random.default_rng(4242))
    rows = []
    for rate in rates:
        low_imu, low_speed = resample(imu, speed, base_rate, rate)          # what the phone records
        grid, grid_speed = resample(low_imu, low_speed, rate, CFG.canonical_rate_hz)
        windows, truth = [], []
        hop = int(CFG.hop_seconds * CFG.canonical_rate_hz)
        for start in range(0, len(grid) - WINDOW, hop):
            windows.append(grid[start:start + WINDOW])
            truth.append(grid_speed[start + WINDOW - 1])
        pred, _ = predict(m, np.asarray(windows, dtype=np.float32))
        rows.append((rate, float(np.mean(pred)), float(np.mean(truth))))
    return rows

rows = rate_invariance(model)
means = [p for _, p, _ in rows]
spread = (max(means) - min(means)) / max(np.mean(means), 1e-6)
print(f"{'rate':>8} {'predicted':>11} {'truth':>9}")
for rate, pred, truth in rows:
    print(f"{rate:8.0f} {pred:11.2f} {truth:9.2f}")
RATE_SPREAD = float(spread)
print(f"\nspread across rates = {spread * 100:.2f} %   (deployed endpoint: 39 %)")
print("PASS" if spread < 0.05 else "FAIL — investigate the resampler before exporting")

### 5c. Turn response

Adding a gentle yaw to a constant-speed cruise must not change the predicted speed much. The
deployed model halved it (18.35 -> 9.42 m/s). With the coordinated-turn residual in the loss, a turn
is *evidence for* the speed, not against it.

In [ ]:
def turn_response(m, speed_mps=16.67, rate_hz=200.0, seconds=20.0):
    results = {}
    for omega in (0.0, 0.05, 0.10, 0.20):
        rng = np.random.default_rng(77)
        n = int(seconds * rate_hz); dt = 1.0 / rate_hz
        speed = np.full(n, speed_mps)
        radius = 0.30
        phase = 2 * math.pi * np.cumsum(speed / (2 * math.pi * radius)) * dt
        vib = np.stack([0.35 * np.sin(phase * (k + 1)) for k in range(3)], axis=-1)
        f = np.stack([np.zeros(n), speed * omega, np.full(n, G)], axis=-1) + vib
        w = np.stack([np.zeros(n), np.zeros(n), np.full(n, omega)], axis=-1)
        R = _rotation(1.1, 0.2, -0.15)
        imu = np.concatenate([f @ R.T, w @ R.T], axis=-1).astype(np.float32)
        imu += rng.normal(0, 0.05, imu.shape).astype(np.float32)
        grid, _ = resample(imu, speed, rate_hz, CFG.canonical_rate_hz)
        windows = np.asarray([grid[s:s + WINDOW] for s in
                              range(0, len(grid) - WINDOW, int(CFG.canonical_rate_hz))], dtype=np.float32)
        pred, _ = predict(m, windows)
        results[omega] = float(np.mean(pred))
    return results

turns = turn_response(model)
straight = turns[0.0]
print(f"{'yaw rate':>10} {'predicted':>11} {'vs straight':>13}")
for omega, value in turns.items():
    print(f"{omega:10.2f} {value:11.2f} {(value / straight - 1) * 100:12.1f} %")
TURN_DEVIATION = max(abs(v / straight - 1) for v in turns.values())
print(f"\nworst deviation = {TURN_DEVIATION * 100:.1f} %   (deployed endpoint: -49 %)")
print("PASS" if TURN_DEVIATION < 0.15 else "FAIL — the physics residual is not binding")

---
## 6. Export for on-device inference

int8 post-training quantisation with a representative dataset drawn from real training windows.
`docs/06` §6.2 is explicit that the parity test is not optional: a 0.1 m/s quantisation bias on the
velocity head is 0.6 % of scale, which is 6 m per km of drift.

In [ ]:
def representative_dataset():
    index = np.random.default_rng(SEED).choice(len(Xtr), size=min(500, len(Xtr)), replace=False)
    for i in index:
        yield [Xtr[i:i + 1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = []
float_tflite = converter.convert()
(OUT / "speed_float.tflite").write_bytes(float_tflite)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
# Keep float in/out: the Android side sends SI units and the 3 ms budget is dominated by the body,
# not by the boundary conversion. Full-integer IO would force the client to know the scale.
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8, tf.lite.OpsSet.TFLITE_BUILTINS]
int8_tflite = converter.convert()
(OUT / "speed_int8.tflite").write_bytes(int8_tflite)

print(f"float  {len(float_tflite) / 1e6:.2f} MB")
print(f"int8   {len(int8_tflite) / 1e6:.2f} MB   (REQ-N2 budget is 8 MB for all heads)")

In [ ]:
def tflite_predict(blob, X, batch_limit=2000):
    interpreter = tf.lite.Interpreter(model_content=blob)
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]
    results = []
    for i in range(min(len(X), batch_limit)):
        interpreter.set_tensor(inp["index"], X[i:i + 1].astype(inp["dtype"]))
        interpreter.invoke()
        results.append(interpreter.get_tensor(out["index"])[0].copy())
    return np.asarray(results)

sample = Xte[:1000]
float_out = tflite_predict(float_tflite, sample)
int8_out = tflite_predict(int8_tflite, sample)
delta = np.abs(float_out[:, 0] - int8_out[:, 0])
PARITY_MAX = float(delta.max()); PARITY_MEAN = float(delta.mean())
print(f"float vs int8 speed: mean {PARITY_MEAN:.4f} m/s, max {PARITY_MAX:.4f} m/s")
print(f"quantisation bias  : {float(np.mean(int8_out[:, 0] - float_out[:, 0])):+.4f} m/s")
PARITY_PASS = PARITY_MAX < 0.25 and abs(np.mean(int8_out[:, 0] - float_out[:, 0])) < 0.05
print("PASS" if PARITY_PASS else "FAIL — retrain with QAT before shipping this head")

In [ ]:
interpreter = tf.lite.Interpreter(model_content=int8_tflite, num_threads=1)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
for _ in range(20):
    interpreter.set_tensor(inp["index"], Xte[:1].astype(inp["dtype"])); interpreter.invoke()
start = time.perf_counter()
REPEATS = 200
for i in range(REPEATS):
    interpreter.set_tensor(inp["index"], Xte[i % len(Xte):i % len(Xte) + 1].astype(inp["dtype"]))
    interpreter.invoke()
LATENCY_MS = (time.perf_counter() - start) / REPEATS * 1000
print(f"single-thread latency on this Colab CPU: {LATENCY_MS:.2f} ms")
print("REQ-N1 is 3 ms on the reference phone; measure there before claiming G-6.")

### Manifest

`docs/05` §5.2 requires the runtime to **refuse to load** on an input-spec mismatch. The spec hash
below is what the Android loader compares against; channel order, units, window length and the
canonical rate are all inside it, so a trainer/runtime disagreement fails loudly at startup rather
than silently producing a wrong speed.

In [ ]:
def sha256(path):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()

input_spec = {
    "channels": ["accel_x", "accel_y", "accel_z", "gyro_x", "gyro_y", "gyro_z"],
    "frame": "phone_body",
    "units": {"accel": "m/s^2 including gravity", "gyro": "rad/s"},
    "window_samples": WINDOW,
    "canonical_rate_hz": CFG.canonical_rate_hz,
    "window_seconds": CFG.window_seconds,
    "resampling": "linear interpolation from the source rate onto the canonical grid",
    "conditioning": "performed inside the graph by the PhysicalFeatures layer; no client-side preprocessing",
}
spec_hash = hashlib.sha256(json.dumps(input_spec, sort_keys=True).encode()).hexdigest()

calibration = {
    "sigma_scale": SIGMA_SCALE,
    "coverage": coverage,
    "gate_g4_pass": bool(G4_PASS),
    "validity_policy": {
        "min_rate_hz": CFG.min_rate_hz,
        "max_rate_hz": CFG.max_rate_hz,
        "max_sigma_mps": 2.5,
        "max_gap_ms": 50.0,
        "notes": "validity is 0 unless every gate passes; it is never a confidence score on its own",
    },
}
(OUT / "calibration.json").write_text(json.dumps(calibration, indent=2))

evaluation = {
    "trained_on": {"real_runs": len(REAL_RUNS), "synthetic_runs": len(SYNTHETIC_RUNS),
                   "source": "IO-VNBD" if REAL_RUNS else "synthetic only"},
    "test_rmse_mps": float(np.sqrt((err ** 2).mean())),
    "test_mae_mps": float(np.abs(err).mean()),
    "median_relative_error_moving": float(np.median(np.abs(err[moving]) / Yte[moving])),
    "coverage": coverage,
    "rate_spread": RATE_SPREAD,
    "turn_worst_deviation": TURN_DEVIATION,
    "int8_parity_max_mps": PARITY_MAX,
    "int8_parity_mean_mps": PARITY_MEAN,
    "colab_cpu_latency_ms": LATENCY_MS,
}
(OUT / "eval_report.json").write_text(json.dumps(evaluation, indent=2))

manifest = {
    "schema": "setu.model-bundle.v1",
    "version": "speed-1.0.0",
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "capabilities": ["speed"],
    "tier_min": "C",
    "vehicles": ["Car"],
    "input_spec": input_spec,
    "input_spec_sha256": spec_hash,
    "quantisation": "int8 post-training, float32 input and output",
    "files": {name: sha256(OUT / name) for name in
              ("speed_int8.tflite", "speed_float.tflite", "calibration.json", "eval_report.json")},
    "evaluation": evaluation,
    "limits": [
        "Trained on synthetic data unless IOVNBD_ROOT was set; check evaluation.trained_on.",
        "No two-wheeler data. Do not enable for VehicleClass.TwoWheeler.",
        "Speed only. This is not a position, and it does not close REQ-P1/P2/P3 on its own.",
        "Latency here is Colab CPU. G-6 needs a measurement on the reference phone.",
    ],
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2)[:1400])

---
## 7. Corrected `setu.model.v1` server

The existing endpoint can be fixed immediately with this, before on-device integration lands. The
differences from what is deployed today are the four defects, in order: it **resamples by
timestamp** rather than trusting sample count, it returns the **calibrated** sigma, it computes a
**real validity** from explicit gates, and it refuses rather than guessing when a gate fails.

It still must not drive navigation until G-4 and G-6 pass on the reference phone. Keeping validity
honest is what lets the app enforce that.

In [ ]:
SERVER = r"""
# Corrected setu.model.v1 inference server for the SETU speed head.
#
# Run:  python model_server.py --bundle setu-speed-v1 --port 8765
import argparse, json, math, pathlib
from http.server import BaseHTTPRequestHandler, HTTPServer

import numpy as np
import tensorflow as tf


class Bundle:
    def __init__(self, root):
        root = pathlib.Path(root)
        self.manifest = json.loads((root / "manifest.json").read_text())
        self.calibration = json.loads((root / "calibration.json").read_text())
        spec = self.manifest["input_spec"]
        self.window = int(spec["window_samples"])
        self.rate = float(spec["canonical_rate_hz"])
        self.seconds = float(spec["window_seconds"])
        self.policy = self.calibration["validity_policy"]
        self.sigma_scale = float(self.calibration["sigma_scale"])
        self.interpreter = tf.lite.Interpreter(
            model_path=str(root / "speed_int8.tflite"), num_threads=2)
        self.interpreter.allocate_tensors()
        self.input = self.interpreter.get_input_details()[0]
        self.output = self.interpreter.get_output_details()[0]

    def infer(self, window):
        self.interpreter.set_tensor(self.input["index"], window.astype(self.input["dtype"]))
        self.interpreter.invoke()
        out = self.interpreter.get_tensor(self.output["index"])[0]
        speed = float(out[0])
        sigma = float(math.exp(0.5 * min(max(out[1], -6.0), 6.0)) * self.sigma_scale)
        return speed, sigma


def prepare(samples, bundle):
    # Resample a timestamped window onto the canonical grid. Returns (window, diagnostics).
    t = np.asarray([s["tNs"] for s in samples], dtype=np.float64)
    if len(t) < 2 or np.any(np.diff(t) <= 0):
        return None, {"reason": "timestamps must be strictly increasing"}
    values = np.asarray(
        [list(s["accelerationMps2"]) + list(s["angularRateRps"]) for s in samples], dtype=np.float64)
    if not np.all(np.isfinite(values)):
        return None, {"reason": "non-finite sample"}

    span_s = (t[-1] - t[0]) / 1e9
    if span_s < bundle.seconds - 0.1:
        return None, {"reason": f"need at least {bundle.seconds:.1f} s of continuous IMU history"}
    measured_rate = (len(t) - 1) / span_s
    gap_ms = float(np.max(np.diff(t)) / 1e6)

    # Resample by TIMESTAMP, not by index. This is the whole fix for the rate defect: the same
    # physical motion at 50 Hz and at 200 Hz lands on the same grid and gives the same answer.
    end = t[-1]
    grid = end - (np.arange(bundle.window)[::-1] / bundle.rate) * 1e9
    window = np.stack([np.interp(grid, t, values[:, c]) for c in range(6)], axis=-1)

    return window[None, ...].astype(np.float32), {
        "measured_rate_hz": measured_rate,
        "max_gap_ms": gap_ms,
        "t_end_ns": int(end),
        "saturated": bool(np.max(np.abs(values[:, :3])) > 78.0 or np.max(np.abs(values[:, 3:])) > 16.0),
    }


def validity(speed, sigma, diagnostics, policy):
    # Zero unless every gate passes. A confidence score is not a validity flag.
    reasons = []
    if not (policy["min_rate_hz"] <= diagnostics["measured_rate_hz"] <= policy["max_rate_hz"]):
        reasons.append("sample rate outside the supported range")
    if diagnostics["max_gap_ms"] > policy["max_gap_ms"]:
        reasons.append("IMU gap in the window")
    if diagnostics["saturated"]:
        reasons.append("sensor saturation")
    if sigma > policy["max_sigma_mps"]:
        reasons.append("predicted uncertainty above the usable bound")
    if not (0.0 <= speed <= 100.0):
        reasons.append("speed outside the contract range")
    if reasons:
        return 0.0, reasons
    # Inside the gates, validity degrades smoothly with the calibrated uncertainty.
    return float(max(0.0, min(1.0, 1.0 - sigma / policy["max_sigma_mps"]))), []


def make_handler(bundle, ready):
    class Handler(BaseHTTPRequestHandler):
        protocol_version = "HTTP/1.1"

        def _send(self, payload, status=200):
            body = json.dumps(payload).encode()
            self.send_response(status)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(body)))
            self.end_headers()
            self.wfile.write(body)

        def do_GET(self):
            if self.path.rstrip("/") != "/v1/health":
                return self._send({"schema": "setu.model.v1", "status": "unavailable",
                                   "reason": "unknown path"}, 404)
            evaluation = bundle.manifest.get("evaluation", {})
            self._send({
                "schema": "setu.model.v1",
                "status": "ready" if ready else "unavailable",
                "model": f"SETU speed {bundle.manifest['version']}",
                "capabilities": bundle.manifest["capabilities"] if ready else [],
                "reason": (
                    "Rate-conditioned, calibrated speed head. "
                    f"3-sigma coverage {evaluation.get('coverage', {}).get('3_sigma', 0) * 100:.1f} %, "
                    f"rate spread {evaluation.get('rate_spread', 0) * 100:.1f} %. "
                    "Evaluation only until G-4 and G-6 pass on the reference phone."
                ),
                "inputSpecSha256": bundle.manifest["input_spec_sha256"],
            })

        def do_POST(self):
            if self.path.rstrip("/") != "/v1/measurements":
                return self._send({"schema": "setu.model.v1", "status": "unavailable",
                                   "reason": "unknown path"}, 404)
            length = int(self.headers.get("Content-Length", 0))
            if not 0 < length <= 8 * 1024 * 1024:
                return self._send({"schema": "setu.model.v1", "status": "unavailable",
                                   "reason": "request body missing or too large"}, 400)
            try:
                request = json.loads(self.rfile.read(length))
            except Exception:
                return self._send({"schema": "setu.model.v1", "status": "unavailable",
                                   "reason": "malformed JSON"}, 400)
            if request.get("schema") != "setu.model.v1":
                return self._send({"schema": "setu.model.v1", "status": "unavailable",
                                   "reason": "unsupported schema"}, 400)
            if request.get("vehicle", "Car") not in bundle.manifest["vehicles"]:
                return self._send({"schema": "setu.model.v1", "status": "unavailable",
                                   "reason": f"unsupported vehicle {request.get('vehicle')!r}"})
            samples = request.get("samples") or []
            if not 1 <= len(samples) <= 2048:
                return self._send({"schema": "setu.model.v1", "status": "unavailable",
                                   "reason": "1-2048 samples required"})
            try:
                window, diagnostics = prepare(samples, bundle)
            except Exception as error:
                return self._send({"schema": "setu.model.v1", "status": "unavailable",
                                   "reason": f"could not read the window: {error}"})
            if window is None:
                return self._send({"schema": "setu.model.v1", "status": "unavailable",
                                   "reason": diagnostics["reason"]})
            speed, sigma = bundle.infer(window)
            value, reasons = validity(speed, sigma, diagnostics, bundle.policy)
            self._send({
                "schema": "setu.model.v1",
                "tNs": diagnostics["t_end_ns"],
                "speedMps": round(speed, 4),
                "sigmaMps": round(max(sigma, 1e-3), 4),
                "validity": round(value, 4),
                "measuredRateHz": round(diagnostics["measured_rate_hz"], 2),
                "gatesFailed": reasons,
            })

        def log_message(self, *args):
            pass

    return Handler


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--bundle", default="setu-speed-v1")
    parser.add_argument("--port", type=int, default=8765)
    parser.add_argument("--host", default="127.0.0.1")
    args = parser.parse_args()
    bundle = Bundle(args.bundle)
    ready = bool(bundle.calibration.get("gate_g4_pass"))
    if not ready:
        print("WARNING: G-4 did not pass in training; serving status=unavailable.")
    print(f"serving {args.host}:{args.port} from {args.bundle}")
    HTTPServer((args.host, args.port), make_handler(bundle, ready)).serve_forever()


if __name__ == "__main__":
    main()
"""

(OUT / "model_server.py").write_text(SERVER.strip() + "\n")
manifest["files"]["model_server.py"] = sha256(OUT / "model_server.py")
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))
print("wrote", OUT / "model_server.py")

---
## 8. Download the bundle

In [ ]:
import shutil
archive = shutil.make_archive("setu-speed-v1", "zip", OUT)
print("bundle:", archive, f"{pathlib.Path(archive).stat().st_size / 1e6:.2f} MB")
for path in sorted(OUT.iterdir()):
    print(f"  {path.name:24s} {path.stat().st_size / 1024:9.1f} KB")

try:
    from google.colab import files
    files.download(archive)
except Exception:
    print("\nNot on Colab - copy setu-speed-v1.zip out manually.")

---
## 9. What this does and does not establish

**Established by this notebook**, assuming the §5 checks printed PASS:

* speed prediction is rate-invariant, which the deployed endpoint is not;
* sigma is heteroscedastic and calibrated, with a coverage number against gate G-4;
* a turn no longer collapses the prediction, because the coordinated-turn residual is in the loss;
* validity is derived from explicit gates instead of being hard-coded;
* the head quantises to int8 within a stated parity bound and fits REQ-N2.

**Not established, and not claimable from this notebook:**

* **REQ-F10 needs IO-VNBD.** If `evaluation.trained_on.source` says `synthetic only`, the numbers
  characterise the pipeline, not real driving. Stage the dataset and rerun before submitting.
* **G-6 needs the reference phone.** Colab CPU latency is not a 3 ms claim on a mid-range
  Snapdragon.
* **G-1/G-2/G-3 are position gates.** A speed head feeds them; it does not close them. That needs
  the native measurement generators in `CHECKPOINT.md` §5 Phase C and a real blackout trial.
* **No two-wheeler support.** There is no lean-form data here; `docs/03` §3.4 explains why a car
  model applied to a leaning vehicle is a silent 4 % scale error.

### Next step for integration

Drop `setu-speed-v1/` into `android/app/src/main/assets/models/`, add the LiteRT dependency, and
implement `ModelProvider` on top of `speed_int8.tflite`. The loader must compare
`input_spec_sha256` and refuse to start on a mismatch (`docs/05` §5.2), and the native core must
gate the measurement on validity, staleness and innovation before it reaches the filter — a valid
JSON response is not an accepted observation.